# Milestone 1 — Colab CPU
Chạy các cell thiết lập theo thứ tự, rồi chọn **một** chế độ crawler. Sau khi kết nối Python mới, mount/chọn lại đúng Drive root, cập nhật code và nạp cấu hình trước khi xem status hoặc resume. Không cần dataset mới.

Bắt buộc có `SEMANTIC_SCHOLAR_API_KEY` trong Colab Secrets, bật Notebook access (hoặc biến môi trường). Không ghi key vào notebook. Không dùng Run all để khởi chạy các chế độ liên tiếp. Sau pull code mới, restart Python nếu crawler đã import, rồi chạy lại phần thiết lập.

## 1. Chọn Drive root và lấy/cập nhật code
Giữ quy trình mount Drive hiện có; chọn thư mục dự án đã mount ở cell dưới.


In [ ]:
from pathlib import Path
import os, subprocess, sys
selected = Path(input('Thư mục dự án đã chọn trên Google Drive: ')).expanduser()
assert selected.is_dir(), 'Chọn thư mục Drive đã mount và tồn tại'
assert input('Xác nhận thư mục này nằm trên Drive (yes): ') == 'yes'
os.chdir(selected)
print('Project root selected')

In [ ]:
repo = Path('code')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/NewB-S2deeC0de/arXiv.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'src/requirements.txt')],check=True)
sys.path.insert(0,str(repo/'src'))

## 2. Nạp cấu hình và kiểm tra key
Nếu thiếu key, bổ sung rồi chạy lại cell này. Cell status vẫn có thể dùng sau lỗi thiếu key vì cấu hình đã nạp.


In [ ]:
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
from crawler.ids import generate_ids
import traceback

config = load_config(Path('code/config/crawler.yaml'))
config.execution['allow_anonymous_s2'] = False

def require_s2_key():
    if not load_semantic_scholar_key():
        raise RuntimeError('Thiếu SEMANTIC_SCHOLAR_API_KEY. Thêm trong Colab Secrets (biểu tượng chìa khóa), bật Notebook access, rồi chạy lại cell này.')

async def run_selected(stages, limit, takeover_stale=False):
    require_s2_key()  # Bắt buộc cả khi chỉ chạy arXiv; không in key.
    config.execution['allow_anonymous_s2'] = False
    try:
        result = await run_pipeline(config, stages, resume=True, limit=limit,
                                    takeover_stale=takeover_stale)
    except BaseException:
        # Hiển thị cả lỗi con của ExceptionGroup ngay trong output cell chạy.
        traceback.print_exc()
        raise
    print(result)
    return result

input_ids = generate_ids(**config.range)
print('Range inclusive:', input_ids[0], '→', input_ids[-1], '| Tổng:', len(input_ids))
print('Persistent root:', config.paths['persistent_root'])
print('Commit:', subprocess.check_output(['git', '-C', 'code', 'log', '-1', '--oneline'], text=True).strip())
require_s2_key()
print('S2 key: đã tìm thấy (chưa kiểm chứng quyền truy cập API)')


## Scratch SSD tùy chọn
Chạy sau cell cấu hình để tải/giải nén trên SSD; result và progress vẫn lưu tại persistent_root trên Drive. Symlink hỏng sau runtime reset được tạo lại. Để False nếu muốn giữ scratch_root trong YAML.


In [ ]:
USE_LOCAL_SCRATCH = True
if USE_LOCAL_SCRATCH:
    import tempfile
    scratch_dir = Path('scratch')
    if scratch_dir.is_symlink() and not scratch_dir.exists():
        scratch_dir.unlink()
    if not scratch_dir.exists():
        scratch_dir.symlink_to(tempfile.mkdtemp(prefix='arxiv-crawler-'), target_is_directory=True)
    assert scratch_dir.is_dir()
    config.paths['scratch_root'] = 'scratch'
    config.validate()
print('Scratch project path:', config.paths['scratch_root'])


## 3. Status — xem sau khi ngắt hoặc kết nối lại
Không gọi mạng API, không reset job hay cooldown. Chạy sau thiết lập; nên chờ cell crawler dừng hẳn để đọc snapshot ổn định. Nếu báo progress hỏng/không đồng bộ, giữ nguyên dữ liệu để kiểm tra, không xóa journal.


In [ ]:
from crawler.progress import ProgressStore
from datetime import datetime, timezone
import json, time

root = Path(config.paths['persistent_root'])
progress = root / 'progress'
if not (progress / 'jobs.csv').exists() and not (progress / 'jobs.bak').exists():
    print('Chưa có snapshot progress tại:', progress)
else:
    # Nạp snapshot + journal mới từ Drive, không dùng store cũ trong RAM.
    store = ProgressStore(root)
    print('Progress:', store.status_counts())
    failures = [r for r in store.rows.values() if r['status'] == 'failed']
    print('Số job failed (gồm page S2):', len(failures))
    for row in failures[:20]:
        print({k: row[k] for k in ('paper', 'version', 'stage', 'offset', 'attempts', 'error')})
    if len(failures) > 20:
        print('Chỉ hiển thị 20 lỗi đầu.')
services = progress / 'services.json'
if services.exists():
    for service, state in json.loads(services.read_text()).items():
        deadline = state.get('deadline', 0)
        print(service, 'cooldown đến:', datetime.fromtimestamp(deadline, timezone.utc).isoformat(),
              '| còn chờ (giây):', round(max(0, deadline - time.time())),
              '| backoff:', state.get('backoff', 0))
print('Status chỉ đọc dữ liệu đã lưu; Drive có thể đồng bộ chậm. Running có thể còn từ phiên bị ngắt.')


## 4. Smoke — tối đa 10 ID đầu
Đổi RUN_SMOKE=True rồi chạy cell. Resume dữ liệu của 10 ID đầu trong range; dữ liệu thành công được bỏ qua nên chạy lại có thể không gọi API. Không tự thử lại failed jobs.


In [ ]:
RUN_SMOKE = False
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_SMOKE:
    summary = await run_selected({'arxiv', 's2'}, limit=10, takeover_stale=TAKEOVER_STALE)
else:
    print('Chưa chạy smoke: đặt RUN_SMOKE=True khi sẵn sàng.')


## 5. Toàn range — arXiv + S2
Tiếp tục toàn range cấu hình, gồm hai nhánh. S2 có thể phải chờ cooldown lâu.
Giữ đúng range/persistent_root cũ để resume. Không reset failed hay cooldown; không chạy đồng thời hai phiên cùng dataset.


In [ ]:
RUN_FULL_RANGE = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_FULL_RANGE:
    summary = await run_selected({'arxiv', 's2'}, limit=None, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_FULL_RANGE=False')


## 6. Chỉ arXiv — toàn range
Chạy discovery, metadata phiên bản và source. Dùng khi S2 đang cooldown; không gửi request S2.
Giữ đúng range/persistent_root cũ để resume. Không reset failed hay cooldown; không chạy đồng thời hai phiên cùng dataset.


In [ ]:
RUN_ARXIV_ONLY = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_ARXIV_ONLY:
    summary = await run_selected({'arxiv'}, limit=None, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_ARXIV_ONLY=False')


## 7. Chỉ Semantic Scholar — toàn range
Chạy references cho dữ liệu đã discovery. Nếu arXiv chưa discovery đủ, chạy arXiv trước rồi chạy lại S2.
Giữ đúng range/persistent_root cũ để resume. Không reset failed hay cooldown; không chạy đồng thời hai phiên cùng dataset.


In [ ]:
RUN_S2_ONLY = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_S2_ONLY:
    summary = await run_selected({'s2'}, limit=None, takeover_stale=TAKEOVER_STALE)
else:
    print('Bỏ qua: RUN_S2_ONLY=False')


## Chẩn đoán source (tùy chọn)
Chỉ xem chẩn đoán, không reset failed. Failed jobs không được tự thử lại khi resume. Xem hướng dẫn retry trong README trước khi reset.


In [ ]:
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))


## Export snapshot (tùy chọn)
Chỉ thực hiện sau khi crawler đã dừng.


In [ ]:
EXPORT_SNAPSHOT = False
if EXPORT_SNAPSHOT:
    from crawler.export import export_snapshot
    import uuid
    print(export_snapshot(config, 'export-' + uuid.uuid4().hex))


### Checkpoint và hiệu năng
Progress gồm jobs.csv và jobs.journal.jsonl; không xóa journal hoặc dùng code cũ sau nâng cấp. Reconcile đầy đủ chạy khi bắt đầu/resume; trong phiên chỉ xác minh result vừa publish. Batch metadata theo IDvN giữ ngày từng phiên bản. Disk metrics dùng baseline + delta. Sau cập nhật, restart kernel, dùng đúng root cũ và smoke/interrupt trước full range.


## Đổi tháng và khoảng ID
Sửa range trong config/crawler.yaml của repo (trên Drive là code/config/crawler.yaml), rồi chạy lại cell load_config. start_month/end_month có dạng YYYY-MM, hỗ trợ khác tháng khi thêm monthly_end_ids cho tháng đầu và từng tháng trung gian; start_id/end_id là số nguyên, hai đầu inclusive. Đây là tháng trong arXiv ID, không phải khoảng ngày/giờ xuất bản bất kỳ. Với dataset mới, đổi paths.persistent_root sang thư mục tương đối mới; giữ root cũ khi resume cùng dataset. Xem README để có ví dụ đầy đủ.


## Dừng trước khi cập nhật code
Stop một lần và chờ cell dừng hẳn. Chưa restart/delete runtime khi Drive chưa đồng bộ; giữ backup toàn runtime tải về máy trước. Xem docs/safe-stop-and-upgrade.md trong repo để có cell backup và kiểm tra snapshot/journal. Giữ nguyên range/root khi resume dataset hiện tại.
